# Hierarchical Deep Learning for Diatom Classification: Google Colab Version

**Author:** Yueying Ke  
**The University of Texas at Austin**

Select a GPU runtime, then run every cell from top to bottom in a fresh Colab session.

# Setup

### Clone the Public Repository

In [ ]:
!git clone https://github.com/DinaberryPi/DiatomCascadeNet-public.git

### Install Dependencies

In [ ]:
%cd /content/DiatomCascadeNet-public
%pip install -q -r requirements.txt
%pip install -q -e . --no-deps

import sys
from pathlib import Path

SRC_DIR = Path.cwd() / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import diatom_cascade
print(f'Package ready: {diatom_cascade.__file__}')

### Prepare and Load Your Dataset

The image dataset used in the original study is not distributed because the project does not hold redistribution rights for the third-party source images. The MIT licence in this repository applies to the code, not to the original image data or annotations. To run the pipeline, obtain appropriately licensed diatom images or provide your own dataset.

Prepare one image per specimen and a metadata workbook with one row per image. This notebook recognizes `metadata_training_clean_from_run.xlsx` (the audited-study workbook) or `metadata.xlsx` (a general dataset workbook). For a complete seven-model run, use these English columns:

| Column | Description |
|---|---|
| `filename` | Exact image filename, including extension |
| `class` | Taxonomic class |
| `order` | Taxonomic order |
| `family` | Taxonomic family |
| `genus` | Taxonomic genus |
| `species` | Taxonomic species |

Filenames in the workbook must exactly match files in `images/`. New datasets should use the English schema above. The converter also accepts the original study's Chinese headers (`ppt序号`, `纲`, `目`, `科`, `属名`, and `种名`) only so the archived workbook can still be read; both schemas are normalized to the same English internal columns.

#### Image requirements

The audited study inputs contained 4,869 RGB PNG files, all exactly `320 x 320` pixels. Therefore, use `IMAGE_SIZE = 320` when reproducing or directly comparing with the study. The value `320` is an experimental setting, not a universal requirement for diatom images. Users exploring their own datasets may choose another square model-input size by changing `IMAGE_SIZE` in `src/diatom_cascade/config/data_config.py` and preparing every final input at that configured size. Results obtained with another size are a new experiment and should not be presented as a strict reproduction of this study.

Raw source images may start at different pixel dimensions. Before this notebook runs, crop each image to one centered diatom specimen, preserve the crop's aspect ratio, and pad it to the configured square size rather than stretching it. For the study setting, prefer source crops of at least `320 x 320`, because enlarging a smaller image cannot recover lost detail. PNG and RGB are recommended; the loader accepts other Pillow-readable formats when the workbook filename matches exactly.

Whichever input size is selected, use it consistently for training, validation, and testing. Keep magnification, illumination, background, color handling, and scale-bar treatment consistent. Remove labels, page text, and borders that could reveal the taxon. If several images come from the same physical specimen or slide, they must not be divided across train, validation, and test; the current manifest builder assumes one independent specimen per filename.

Prepare this private Google Drive layout:

```text
MyDrive/DiatomScanNet/
  images/
  metadata_training_clean_from_run.xlsx  # preferred for the audited study
  # or metadata.xlsx for another dataset
  invalid_images.csv  # optional
```

`invalid_images.csv` is optional. When supplied, it must contain a `filename` column listing images that should be excluded; additional audit columns are allowed. If it is absent, the notebook creates an empty exclusion manifest. Later preflight checks still stop on missing, unreadable, uniform, or duplicate images. These private files stay in the Colab runtime and Google Drive; they are ignored by Git.

In [ ]:
from pathlib import Path
import shutil

from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/DiatomCascadeNet-public')
DATA_ROOT = PROJECT_ROOT / 'dataset'
DRIVE_ROOT = Path('/content/drive/MyDrive/DiatomScanNet')

metadata_candidates = [
    DRIVE_ROOT / 'metadata_training_clean_from_run.xlsx',
    DRIVE_ROOT / 'metadata.xlsx',
]
metadata_source = next((path for path in metadata_candidates if path.is_file()), None)
required_inputs = [DRIVE_ROOT / 'images']
missing = [str(path) for path in required_inputs if not path.exists()]
if missing or metadata_source is None:
    if metadata_source is None:
        missing.append('metadata_training_clean_from_run.xlsx or metadata.xlsx')
    raise FileNotFoundError(f'Missing Google Drive inputs: {missing}')

shutil.copytree(
    DRIVE_ROOT / 'images',
    DATA_ROOT / 'raw' / 'images',
    dirs_exist_ok=True,
)
shutil.copy2(metadata_source, DATA_ROOT / 'raw' / 'metadata.xlsx')
exclusion_source = DRIVE_ROOT / 'invalid_images.csv'
exclusion_target = DATA_ROOT / 'exclusions' / 'invalid_images.csv'
if exclusion_source.is_file():
    shutil.copy2(exclusion_source, exclusion_target)
else:
    exclusion_target.write_text('filename\n', encoding='utf-8')
    print('No exclusion manifest supplied; using an empty manifest.')

image_files = [
    path for path in (DATA_ROOT / 'raw' / 'images').iterdir()
    if path.is_file() and not path.name.startswith('.')
]
if not image_files:
    raise FileNotFoundError(f'No images found in {DATA_ROOT / "raw" / "images"}')

print(f'Inputs ready: {len(image_files)} images; metadata: {metadata_source.name}')

### Initialize the Run

Change `RUN_ID` for every new experiment. The run folder is created only after data preparation and validation succeed.

In [ ]:
import os

RUN_ID = 'colab_2026_r01'
OUTPUT_DIR = Path('/content/drive/MyDrive/DiatomScanNet/runs') / RUN_ID

if OUTPUT_DIR.exists():
    raise FileExistsError(f'Choose a new RUN_ID: {OUTPUT_DIR} already exists')

os.environ['DIATOM_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['DIATOM_DATA_ROOT'] = str(DATA_ROOT)
os.environ['DIATOM_OUTPUT_DIR'] = str(OUTPUT_DIR)

print(f'Run: {RUN_ID}')
print(f'Output: {OUTPUT_DIR}')

# 1. Data Preparation

### Run Software Preflight Tests

The subprocess helper stops the notebook immediately when any script fails. Each pipeline stage runs in a clean Python process.

In [ ]:
import subprocess
import sys
import time
from datetime import timedelta

import torch

def run_module(module, *args):
    print(f'\n=== {module} ===', flush=True)
    started = time.perf_counter()
    completed = subprocess.run(
        [sys.executable, '-m', module, *map(str, args)],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, end='')
    if completed.returncode:
        if completed.stderr:
            print(completed.stderr, end='', file=sys.stderr)
        completed.check_returncode()
    elapsed = timedelta(seconds=round(time.perf_counter() - started))
    print(f'SUCCESS: {module} completed in {elapsed}', flush=True)

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before starting the experiment')
print(f'GPU: {torch.cuda.get_device_name(0)}')

run_module('unittest', 'discover', '-s', 'tests', '-p', 'test_*.py', '-v')

### Build Clean Datasets and Fixed Splits

This converts the supplied metadata, applies the explicit exclusion manifest, builds the deterministic taxonomy tree, creates model-specific filtered datasets, checks every referenced image, and writes fixed train/validation/test manifests.

In [ ]:
data_modules = [
    'scripts.data.labeling.convert_metadata',
    'scripts.data.cleaning.clean_data',
    'scripts.data.preprocessing.build_taxonomy_tree',
    'scripts.data.preprocessing.create_filtered_datasets',
    'scripts.data.preprocessing.create_split_manifests',
]

for module in data_modules:
    run_module(module)

### Verify All Train/Validation/Test Manifests

The two controlled comparison pairs must have identical rows in every split: H-COFG versus F-G, and H-COFGS versus F-S.

In [ ]:
import pandas as pd

from diatom_cascade.data.integrity import load_split_manifests

model_names = ['F-C', 'H-CO', 'H-COF', 'H-COFG', 'H-COFGS', 'F-G', 'F-S']
split_frames = {}
split_counts = {}

for model_name in model_names:
    frames = load_split_manifests(DATA_ROOT, model_name)
    split_frames[model_name] = frames
    split_counts[model_name] = {
        name: len(frame)
        for name, frame in zip(('train', 'validation', 'test'), frames)
    }
    print(model_name, split_counts[model_name])

for left, right in (('H-COFG', 'F-G'), ('H-COFGS', 'F-S')):
    for split_name, left_frame, right_frame in zip(
        ('train', 'validation', 'test'),
        split_frames[left],
        split_frames[right],
    ):
        pd.testing.assert_frame_equal(
            left_frame.sort_values('filename').reset_index(drop=True),
            right_frame.sort_values('filename').reset_index(drop=True),
            check_dtype=False,
            obj=f'{left} vs {right} {split_name}',
        )

print('All seven manifest sets and both controlled comparison pairs passed.')

### Record the Run Environment

This cell creates the run folder, archives the exact split manifests, and records the source commit, software versions, configuration, and SHA-256 hashes of every prepared input.

In [ ]:
import hashlib
import json
import platform
from datetime import datetime, timezone

import pandas as pd
import torch

from diatom_cascade.checkpoints import CHECKPOINT_SCHEMA_VERSION
from diatom_cascade.config.train_and_val_config import TrainAndValConfig

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

git_status = subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True
)
if git_status.strip():
    raise RuntimeError('Commit source changes before starting a run')

evidence_files = [
    DATA_ROOT / 'raw' / 'metadata.xlsx',
    DATA_ROOT / 'raw' / 'labels.csv',
    DATA_ROOT / 'exclusions' / 'invalid_images.csv',
    DATA_ROOT / 'cleaned' / 'labels_clean.csv',
    *sorted((DATA_ROOT / 'preprocessed').glob('*.csv')),
    *sorted((DATA_ROOT / 'preprocessed').glob('*.json')),
    *sorted((DATA_ROOT / 'splits').rglob('*.csv')),
]
missing = [str(path) for path in evidence_files if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing prepared artifacts: {missing}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
shutil.copytree(DATA_ROOT / 'splits', OUTPUT_DIR / 'split_manifests')
image_paths = sorted(
    path for path in (DATA_ROOT / 'raw' / 'images').iterdir()
    if path.is_file() and not path.name.startswith('.')
)
image_inventory_path = OUTPUT_DIR / 'image_inventory.csv'
pd.DataFrame([
    {
        'filename': path.name,
        'sha256': sha256(path),
        'size_bytes': path.stat().st_size,
    }
    for path in image_paths
]).to_csv(image_inventory_path, index=False)

manifest = {
    'run_id': RUN_ID,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True
    ).strip(),
    'python': platform.python_version(),
    'pytorch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'checkpoint_schema_version': CHECKPOINT_SCHEMA_VERSION,
    'image_inventory_sha256': sha256(image_inventory_path),
    'split_counts': split_counts,
    'training_config': {
        'random_seed': TrainAndValConfig.RANDOM_SEED,
        'image_size': TrainAndValConfig.IMAGE_SIZE,
        'batch_size': TrainAndValConfig.BATCH_SIZE,
        'base_model': TrainAndValConfig.BASE_MODEL,
        'backbone_pretrain': TrainAndValConfig.BACKBONE_PRETRAIN,
        'optimizer': TrainAndValConfig.OPTIMIZER,
        'initial_lr': TrainAndValConfig.INITIAL_LR,
        'weight_decay': TrainAndValConfig.WEIGHT_DECAY,
        'max_epochs': TrainAndValConfig.MAX_EPOCHS,
        'minimum_samples': TrainAndValConfig.MIN_SAMPLES,
    },
    'input_hashes': {
        str(path.relative_to(DATA_ROOT)): sha256(path) for path in evidence_files
    },
}

with (OUTPUT_DIR / 'run_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=True)
with (OUTPUT_DIR / 'pip_freeze.txt').open('w', encoding='utf-8') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=handle, check=True)

print(f'Recorded {len(image_paths)} image hashes and {len(evidence_files)} prepared-input hashes')

# 2. Model Training

Run the model cells in the displayed order. F-C initializes the progressive chain. H-CO, H-COF, H-COFG, and H-COFGS transfer only the trained backbone from the preceding stage and initialize new classifier heads. F-G and F-S are independent flat baselines. Separate cells prevent a late failure from restarting earlier completed models.

### F-C

In [ ]:
run_module('scripts.train.train_F_C')

### H-CO

In [ ]:
run_module('scripts.train.train_H_CO')

### H-COF

In [ ]:
run_module('scripts.train.train_H_COF')

### H-COFG

In [ ]:
run_module('scripts.train.train_H_COFG')

### H-COFGS

In [ ]:
run_module('scripts.train.train_H_COFGS')

### F-G Baseline

In [ ]:
run_module('scripts.train.train_F_G')

### F-S Baseline

In [ ]:
run_module('scripts.train.train_F_S')

# 3. Evaluation

All models are evaluated on their archived test manifests. Greedy hierarchical prediction is the primary reported method. Argmax and beam-search diagnostics are retained in the JSON reports but must not be presented as primary paper results without separate validation.

In [ ]:
run_module('scripts.evaluate.run_all_evaluations')

expected_reports = [
    OUTPUT_DIR / 'evaluation' / f'{name}_evaluation_report.json'
    for name in ('F_C', 'F_G', 'F_S', 'H_CO', 'H_COF', 'H_COFG', 'H_COFGS')
]
missing = [str(path) for path in expected_reports if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing evaluation reports: {missing}')
print('All seven evaluation reports are present.')

# 4. Paper Analysis

### Primary Species-Level Error-Propagation Analysis

This is the paper-facing H-COFGS versus F-S comparison. It uses predictions from the same species-level test manifest.

In [ ]:
run_module(
    'scripts.experiments.error_propagation.error_propagation_H_COFGS_vs_F_S'
)

species_error_results = (
    OUTPUT_DIR
    / 'figures'
    / 'error_propagation'
    / 'error_propagation_H_COFGS_vs_F_S_results.json'
)
if not species_error_results.is_file():
    raise FileNotFoundError(f'Error-propagation analysis did not finish: {species_error_results}')

### Supporting Genus-Level Analysis

This H-COFG versus F-G comparison is retained as a reference analysis. It is not the paper's primary error-propagation result.

In [ ]:
run_module(
    'scripts.experiments.error_propagation.error_propagation_H_COFG_vs_F_G'
)

genus_error_results = (
    OUTPUT_DIR
    / 'figures'
    / 'error_propagation'
    / 'error_propagation_H_COFG_vs_F_G_REFERENCE_results.json'
)
if not genus_error_results.is_file():
    raise FileNotFoundError(f'Genus reference analysis did not finish: {genus_error_results}')

### Generate Tables and Figures

In [ ]:
analysis_modules = [
    'scripts.analysis.generate_all_results_table',
    'scripts.analysis.plot_pipeline_pyramids',
    'scripts.analysis.training_curves.plot_F_C_training_curves',
    'scripts.analysis.training_curves.plot_H_CO_training_curves',
    'scripts.analysis.training_curves.plot_H_COF_training_curves',
    'scripts.analysis.training_curves.plot_H_COFG_training_curves',
    'scripts.analysis.training_curves.plot_H_COFGS_training_curves',
    'scripts.analysis.training_curves.plot_F_G_training_curves',
    'scripts.analysis.training_curves.plot_F_S_training_curves',
    'scripts.analysis.eval_figures.plot_progressive_methods_comparison',
    'scripts.analysis.eval_figures.plot_H_COFG_vs_F_G',
    'scripts.analysis.eval_figures.plot_H_COFGS_vs_F_S',
    'scripts.analysis.eval_figures.plot_error_propagation',
]

for module in analysis_modules:
    run_module(module)

# 5. Finalize the Run

The final cell verifies the essential checkpoints, reports, and paper-analysis files before writing artifact hashes and a completion marker.

In [ ]:
expected_checkpoints = [
    OUTPUT_DIR / 'checkpoints' / f'best_{name}_model.pth'
    for name in ('F_C', 'F_G', 'F_S', 'H_CO', 'H_COF', 'H_COFG', 'H_COFGS')
]
essential_artifacts = [
    OUTPUT_DIR / 'run_manifest.json',
    OUTPUT_DIR / 'pip_freeze.txt',
    OUTPUT_DIR / 'all_results_tables.md',
    species_error_results,
    genus_error_results,
    *expected_checkpoints,
    *expected_reports,
]
missing = [str(path) for path in essential_artifacts if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Run is incomplete; missing artifacts: {missing}')

artifacts = sorted(path for path in OUTPUT_DIR.rglob('*') if path.is_file())
artifact_hashes = {
    str(path.relative_to(OUTPUT_DIR)): sha256(path) for path in artifacts
}
with (OUTPUT_DIR / 'artifact_hashes.json').open('w', encoding='utf-8') as handle:
    json.dump(artifact_hashes, handle, indent=2, ensure_ascii=True)
with (OUTPUT_DIR / 'RUN_COMPLETE.txt').open('w', encoding='utf-8') as handle:
    handle.write(datetime.now(timezone.utc).isoformat() + '\n')

print(f'Completed {RUN_ID}')
print(f'Artifacts: {len(artifacts)}')
print(f'Output: {OUTPUT_DIR}')